In [1]:
import os, glob, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from scipy.signal import savgol_filter
from scipy.stats import pearsonr
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

try:
    from dtaidistance import dtw as dtaidtw
    DTW_AVAILABLE = True
    print('dtw ok')
except:
    DTW_AVAILABLE = False
    print('no dtw - pip install dtaidistance')

def rmse(y, yhat):
    return np.sqrt(mean_squared_error(y, yhat))

no dtw - pip install dtaidistance


In [2]:
class CFG:
    TRAIN_DIR = '/kaggle/input/competitions/rogii-wellbore-geology-prediction/train'
    TEST_DIR  = '/kaggle/input/competitions/rogii-wellbore-geology-prediction/test'
    SUB_FILE  = '/kaggle/working/submission.csv'
    SEED      = 42
    N_FOLDS   = 5
    ROLL_WINDOWS = [5, 10, 20, 50, 100]
    LGB_PARAMS = dict(
        objective        = 'regression',
        metric           = 'rmse',
        num_leaves       = 255,
        learning_rate    = 0.05,
        feature_fraction = 0.8,
        bagging_fraction = 0.8,
        bagging_freq     = 5,
        min_child_samples= 50,
        n_estimators     = 5000,
        early_stopping_rounds = 100,
        verbose          = -1,
        n_jobs           = -1,
        random_state     = SEED,
    )

In [3]:
# load all wells from a directory
def load_well(well_dir, well_name):
    hz = pd.read_csv(os.path.join(well_dir, f'{well_name}__horizontal_well.csv'))
    tw = pd.read_csv(os.path.join(well_dir, f'{well_name}__typewell.csv'))
    hz['well_id'] = well_name
    tw['well_id'] = well_name
    return hz, tw

def load_all_wells(data_dir, is_train=True):
    hz_files   = sorted(glob.glob(os.path.join(data_dir, '*__horizontal_well.csv')))
    well_names = [Path(f).name.replace('__horizontal_well.csv', '') for f in hz_files]
    all_hz, all_tw = [], []
    for wn in tqdm(well_names, desc=f'loading {"train" if is_train else "test"}'):
        try:
            hz, tw = load_well(data_dir, wn)
            all_hz.append(hz)
            all_tw.append(tw)
        except Exception as e:
            print(f'  skip {wn}: {e}')
    return pd.concat(all_hz, ignore_index=True), pd.concat(all_tw, ignore_index=True)

train_hz, train_tw = load_all_wells(CFG.TRAIN_DIR, is_train=True)
test_hz,  test_tw  = load_all_wells(CFG.TEST_DIR,  is_train=False)

print(f'train: {train_hz.well_id.nunique()} wells, {len(train_hz):,} rows')
print(f'test:  {test_hz.well_id.nunique()} wells,  {len(test_hz):,} rows')

loading train:   0%|          | 0/773 [00:00<?, ?it/s]

loading test:   0%|          | 0/3 [00:00<?, ?it/s]

train: 773 wells, 5,092,255 rows
test:  3 wells,  19,221 rows


In [4]:
# DTW: align horizontal GR to typewell GR, map back to TVT
def dtw_align_tvt(hz_gr, tw_gr, tw_tvt):
    if not DTW_AVAILABLE:
        return np.full(len(hz_gr), np.nan)
    hz_n = (hz_gr - hz_gr.mean()) / (hz_gr.std() + 1e-8)
    tw_n = (tw_gr - tw_gr.mean()) / (tw_gr.std() + 1e-8)
    try:
        path     = dtaidtw.warping_path(hz_n.astype(np.float64), tw_n.astype(np.float64))
        path_arr = np.array(path)
        out      = np.full(len(hz_gr), np.nan)
        for i in range(len(hz_gr)):
            matched = path_arr[path_arr[:, 0] == i, 1]
            if len(matched):
                out[i] = tw_tvt[min(int(np.median(matched)), len(tw_tvt) - 1)]
        mask = np.isnan(out)
        if mask.any():
            idx = np.arange(len(out))
            out[mask] = np.interp(idx[mask], idx[~mask], out[~mask])
        return out
    except Exception as e:
        print(f'  dtw failed: {e}')
        return np.linspace(tw_tvt[0], tw_tvt[-1], len(hz_gr))

In [5]:
def build_features(hz_df, tw_df):
    records = []

    for wid in tqdm(hz_df['well_id'].unique(), desc='features'):
        hz = hz_df[hz_df['well_id'] == wid].sort_values('MD').reset_index(drop=True).copy()
        tw = tw_df[tw_df['well_id'] == wid].sort_values('TVT').copy()

        hz_gr  = hz['GR'].fillna(hz['GR'].median()).values
        tw_gr  = tw['GR'].fillna(tw['GR'].median()).values
        tw_tvt = tw['TVT'].values

        # DTW-predicted TVT
        hz['dtw_tvt_pred'] = dtw_align_tvt(hz_gr, tw_gr, tw_tvt)

        # propagate known TVT_input forward and backward
        # this gives context about where the eval zone sits relative to known geology
        known = hz['TVT_input'].values.copy()

        fwd = np.full(len(hz), np.nan)
        last = np.nan
        for i in range(len(hz)):
            if not np.isnan(known[i]): last = known[i]
            fwd[i] = last

        bwd = np.full(len(hz), np.nan)
        nxt = np.nan
        for i in range(len(hz) - 1, -1, -1):
            if not np.isnan(known[i]): nxt = known[i]
            bwd[i] = nxt

        hz['tvt_fwd']    = fwd
        hz['tvt_bwd']    = bwd
        hz['tvt_interp'] = (np.nan_to_num(fwd) + np.nan_to_num(bwd)) / 2

        # TVT rate of change from the known portion
        known_mask = ~np.isnan(known)
        if known_mask.sum() > 1:
            ki  = np.where(known_mask)[0]
            kv  = known[known_mask]
            grad = np.gradient(kv, ki)
            hz['tvt_grad'] = np.interp(np.arange(len(hz)), ki, grad)
        else:
            hz['tvt_grad'] = 0.0

        # typewell summary stats
        hz['tw_gr_mean']   = tw_gr.mean()
        hz['tw_gr_std']    = tw_gr.std()
        hz['tw_tvt_mean']  = tw_tvt.mean()
        hz['tw_tvt_range'] = tw_tvt.max() - tw_tvt.min()
        hz['tw_tvt_min']   = tw_tvt.min()
        hz['tw_tvt_max']   = tw_tvt.max()
        hz['gr_vs_tw']     = hz_gr - tw_gr.mean()

        # nearest typewell GR match → TVT
        # for each depth step find the typewell row with closest GR value
        gr_diffs = np.abs(hz_gr[:, None] - tw_gr[None, :])
        best_idx = np.argmin(gr_diffs, axis=1)
        hz['tw_nearest_tvt'] = tw_tvt[best_idx]
        top3_idx = np.argpartition(gr_diffs, min(3, gr_diffs.shape[1] - 1), axis=1)[:, :3]
        hz['tw_top3_tvt_mean'] = tw_tvt[top3_idx].mean(axis=1)

        # GR cross-correlation with typewell
        n = min(len(hz_gr), len(tw_gr))
        hz_n = (hz_gr[:n] - hz_gr[:n].mean()) / (hz_gr[:n].std() + 1e-8)
        tw_n = (tw_gr[:n] - tw_gr[:n].mean()) / (tw_gr[:n].std() + 1e-8)
        try:    corr, _ = pearsonr(hz_n, tw_n)
        except: corr = 0.0
        hz['xcorr'] = corr

        records.append(hz)

    df = pd.concat(records, ignore_index=True).sort_values(['well_id', 'MD'])

    # rolling GR stats
    for w in CFG.ROLL_WINDOWS:
        g = df.groupby('well_id')['GR']
        df[f'GR_mean_{w}']  = g.transform(lambda x: x.rolling(w, min_periods=1, center=True).mean())
        df[f'GR_std_{w}']   = g.transform(lambda x: x.rolling(w, min_periods=1, center=True).std().fillna(0))
        df[f'GR_range_{w}'] = (
            g.transform(lambda x: x.rolling(w, min_periods=1, center=True).max()) -
            g.transform(lambda x: x.rolling(w, min_periods=1, center=True).min())
        )

    df['GR_grad']   = df.groupby('well_id')['GR'].transform(lambda x: np.gradient(x.values))
    df['GR_zscore'] = df.groupby('well_id')['GR'].transform(lambda x: (x - x.mean()) / (x.std() + 1e-8))

    # spatial
    df['md_from_heel'] = df.groupby('well_id')['MD'].transform(lambda x: x - x.min())
    df['md_rel']       = df.groupby('well_id')['MD'].transform(
        lambda x: (x - x.min()) / (x.max() - x.min() + 1e-8))
    df['dZ'] = df.groupby('well_id')['Z'].transform(lambda x: x.diff().fillna(0))

    return df

print('building train features...')
train_feat = build_features(train_hz, train_tw)
print('building test features...')
test_feat  = build_features(test_hz,  test_tw)

building train features...


features:   0%|          | 0/773 [00:00<?, ?it/s]

building test features...


features:   0%|          | 0/3 [00:00<?, ?it/s]

In [6]:
# TVT_input notna  → training rows (geology is known here)
# TVT_input isna   → eval zone we need to predict
train_df      = train_feat[train_feat['TVT_input'].notna()].copy()
pred_test_df  = test_feat[test_feat['TVT_input'].isna()].copy()

print(f'train rows:      {len(train_df):,}')
print(f'test pred rows:  {len(pred_test_df):,}')

# drop cols that don't exist in test (ANCC, EGFDU etc. are train-only geologic surfaces)
EXCLUDE = {'well_id', 'TVT', 'TVT_input', 'is_train', 'Geology',
           'ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA'}

feature_cols = [
    c for c in train_df.columns
    if c not in EXCLUDE
    and c in pred_test_df.columns
    and train_df[c].dtype in [np.float64, np.float32, np.int64, np.int32]
]

train_df[feature_cols]     = train_df[feature_cols].fillna(0)
pred_test_df[feature_cols] = pred_test_df[feature_cols].fillna(0)

print(f'features: {len(feature_cols)}')
print(feature_cols)

train rows:      1,308,266
test pred rows:  14,151
features: 40
['MD', 'X', 'Y', 'Z', 'GR', 'dtw_tvt_pred', 'tvt_fwd', 'tvt_bwd', 'tvt_interp', 'tvt_grad', 'tw_gr_mean', 'tw_gr_std', 'tw_tvt_mean', 'tw_tvt_range', 'tw_tvt_min', 'tw_tvt_max', 'gr_vs_tw', 'tw_nearest_tvt', 'tw_top3_tvt_mean', 'xcorr', 'GR_mean_5', 'GR_std_5', 'GR_range_5', 'GR_mean_10', 'GR_std_10', 'GR_range_10', 'GR_mean_20', 'GR_std_20', 'GR_range_20', 'GR_mean_50', 'GR_std_50', 'GR_range_50', 'GR_mean_100', 'GR_std_100', 'GR_range_100', 'GR_grad', 'GR_zscore', 'md_from_heel', 'md_rel', 'dZ']


In [7]:
# quick sanity check: DTW alone vs naive mean
if 'dtw_tvt_pred' in train_df.columns:
    y_true = train_df['TVT'].values
    print(f'DTW alone RMSE:  {rmse(y_true, train_df["dtw_tvt_pred"].fillna(0).values):.4f}')
    print(f'naive mean RMSE: {rmse(y_true, np.full_like(y_true, y_true.mean())):.4f}')

DTW alone RMSE:  11401.4392
naive mean RMSE: 663.1709


In [8]:
X      = train_df[feature_cols].values
y      = train_df['TVT'].values
groups = train_df['well_id'].values

lgb_models    = []
oof_rmse_list = []
gkf           = GroupKFold(n_splits=CFG.N_FOLDS)

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    print(f'\n--- fold {fold+1}/{CFG.N_FOLDS} ---')
    model = lgb.LGBMRegressor(**CFG.LGB_PARAMS)
    model.fit(
        X[tr_idx], y[tr_idx],
        eval_set=[(X[val_idx], y[val_idx])],
        callbacks=[
            lgb.early_stopping(CFG.LGB_PARAMS['early_stopping_rounds'], verbose=False),
            lgb.log_evaluation(200),
        ],
    )
    fold_rmse = rmse(y[val_idx], model.predict(X[val_idx]))
    oof_rmse_list.append(fold_rmse)
    lgb_models.append(model)
    print(f'fold rmse: {fold_rmse:.4f}')

print(f'\nmean OOF RMSE: {np.mean(oof_rmse_list):.4f}')


--- fold 1/5 ---
[200]	valid_0's rmse: 8.36517
[400]	valid_0's rmse: 8.31982
fold rmse: 8.3192

--- fold 2/5 ---
[200]	valid_0's rmse: 8.26265
[400]	valid_0's rmse: 8.22141
[600]	valid_0's rmse: 8.20792
[800]	valid_0's rmse: 8.20598
fold rmse: 8.2049

--- fold 3/5 ---
[200]	valid_0's rmse: 7.47708
[400]	valid_0's rmse: 7.44073
[600]	valid_0's rmse: 7.43708
fold rmse: 7.4325

--- fold 4/5 ---
[200]	valid_0's rmse: 9.1727
[400]	valid_0's rmse: 9.11998
fold rmse: 9.1053

--- fold 5/5 ---
[200]	valid_0's rmse: 5.12839
[400]	valid_0's rmse: 5.08229
[600]	valid_0's rmse: 5.06207
[800]	valid_0's rmse: 5.0457
[1000]	valid_0's rmse: 5.03511
[1200]	valid_0's rmse: 5.0254
[1400]	valid_0's rmse: 5.02084
[1600]	valid_0's rmse: 5.01875
[1800]	valid_0's rmse: 5.01591
[2000]	valid_0's rmse: 5.01645
fold rmse: 5.0150

mean OOF RMSE: 7.6154


In [9]:
# feature importance — useful to understand what's actually driving predictions
imp = np.mean([m.feature_importances_ for m in lgb_models], axis=0)
fi  = pd.DataFrame({'feature': feature_cols, 'importance': imp})
print(fi.sort_values('importance', ascending=False).head(20).to_string(index=False))

     feature  importance
     tvt_fwd     24532.8
md_from_heel     12457.4
      md_rel     10691.0
          MD      8898.4
  GR_std_100      7893.4
           Z      7669.8
 GR_mean_100      7347.6
    tvt_grad      7333.0
  tw_tvt_min      6647.8
   GR_std_50      6398.6
GR_range_100      6072.2
  GR_mean_50      5927.4
 GR_range_50      5910.8
          dZ      5685.8
 GR_range_20      5622.6
     tvt_bwd      5321.6
   GR_std_20      5148.4
  GR_mean_20      4984.2
           X      4449.8
       xcorr      4380.2


In [10]:
# average predictions across all folds
X_test     = pred_test_df[feature_cols].values
test_preds = np.mean([m.predict(X_test) for m in lgb_models], axis=0)

pred_test_df = pred_test_df.reset_index(drop=True).copy()
pred_test_df['TVT_pred'] = test_preds

# Savitzky-Golay smoothing per well
# TVT should be physically smooth — this mimics what a geologist would do
def smooth_per_well(df, col='TVT_pred', window=11, poly=3):
    out = df[col].values.copy()
    for _, grp in df.groupby('well_id'):
        idx  = grp.index.values
        vals = out[idx]
        if len(vals) > window:
            try:    out[idx] = savgol_filter(vals, window_length=window, polyorder=poly)
            except: pass
    return out

pred_test_df['TVT_pred'] = smooth_per_well(pred_test_df)
print(f'predictions done — shape: {pred_test_df.shape}')
print(pred_test_df[['well_id', 'MD', 'TVT_pred']].head())

predictions done — shape: (14151, 43)
    well_id       MD      TVT_pred
0  000d7d20  12909.0  11370.743636
1  000d7d20  12910.0  11370.482200
2  000d7d20  12911.0  11370.465150
3  000d7d20  12912.0  11370.690554
4  000d7d20  12913.0  11371.156480


In [11]:
# build submission in the exact format the competition expects
sample_path = '/kaggle/input/competitions/rogii-wellbore-geology-prediction/sample_submission.csv'
sample      = pd.read_csv(sample_path)
print('sample format:')
print(sample.head())
print(sample.columns.tolist())

sample format:
              id  tvt
0  000d7d20_1442  0.0
1  000d7d20_1443  0.0
2  000d7d20_1444  0.0
3  000d7d20_1445  0.0
4  000d7d20_1446  0.0
['id', 'tvt']


In [12]:
# id format is  {wellname}_{row_index}  e.g. 015fe0d2_0, 015fe0d2_1 ...
pred_test_df['row_index'] = pred_test_df.groupby('well_id').cumcount()
pred_test_df['id']        = pred_test_df['well_id'] + '_' + pred_test_df['row_index'].astype(str)

# figure out the target column name from sample
target_col = [c for c in sample.columns if c != 'id'][0]
print(f'target col: {target_col}')

sub = sample[['id']].merge(
    pred_test_df[['id', 'TVT_pred']].rename(columns={'TVT_pred': target_col}),
    on='id', how='left'
)
# fill any unmatched rows with median (shouldn't happen but just in case)
sub[target_col] = sub[target_col].fillna(sub[target_col].median())

sub.to_csv(CFG.SUB_FILE, index=False)
print(f'saved → {CFG.SUB_FILE}')
print(f'shape: {sub.shape}')
print(sub.head(10))

target col: tvt
saved → /kaggle/working/submission.csv
shape: (14151, 2)
              id           tvt
0  000d7d20_1442  11378.220936
1  000d7d20_1443  11378.253667
2  000d7d20_1444  11378.314569
3  000d7d20_1445  11378.402788
4  000d7d20_1446  11378.446837
5  000d7d20_1447  11378.437310
6  000d7d20_1448  11378.380005
7  000d7d20_1449  11378.287223
8  000d7d20_1450  11378.129170
9  000d7d20_1451  11378.029387
